# IPL Cricket Data Analysis — SQL (SQLite) + Tableau

**Author:** Galib Amin  |  **Tools:** Python, SQLite, SQL (window functions, CTEs, CASE WHEN), Matplotlib, Tableau Public

**Dataset:** [IPL Complete Dataset 2008-2024 (Cricsheet via Kaggle)](https://www.kaggle.com/datasets/patrickb1912/ipl-complete-dataset-20082020) — `matches.csv` (one row per match) and `deliveries.csv` (one row per ball).

### Project flow
1. Load and clean the raw CSVs with pandas
2. Load them into a **SQLite** database and build an analysis view
3. Answer **28 business questions with SQL** (team, batting, bowling, venue and match-level insights)
4. Visualise key findings with Matplotlib
5. Export tidy CSVs for the **Tableau Public** dashboard

## 1. Setup and data loading

In [ ]:
%pip install -q kagglehub

In [ ]:
import os, glob, sqlite3, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

os.makedirs("images", exist_ok=True)
os.makedirs("tableau_exports", exist_ok=True)

In [ ]:
# ---- Locate matches.csv and deliveries.csv ---------------------------------
# Order of attempts: (1) files already in the Colab session, (2) download from Kaggle, (3) manual upload.
def find_csvs(root="."):
    m = glob.glob(os.path.join(root, "**", "matches*.csv"), recursive=True)
    d = glob.glob(os.path.join(root, "**", "deliveries*.csv"), recursive=True)
    return (m[0] if m else None), (d[0] if d else None)

m_path, d_path = find_csvs(".")

if not (m_path and d_path):
    try:
        import kagglehub
        folder = kagglehub.dataset_download("patrickb1912/ipl-complete-dataset-20082020")
        m_path, d_path = find_csvs(folder)
        print("Downloaded from Kaggle:", folder)
    except Exception as e:
        print("Kaggle download failed ->", type(e).__name__, "| falling back to manual upload")

if not (m_path and d_path):
    from google.colab import files
    print("Please upload matches.csv and deliveries.csv")
    files.upload()
    m_path, d_path = find_csvs(".")

print("matches   :", m_path)
print("deliveries:", d_path)

In [ ]:
matches    = pd.read_csv(m_path)
deliveries = pd.read_csv(d_path)

print("matches   :", matches.shape)
print("deliveries:", deliveries.shape)
matches.head(3)

In [ ]:
deliveries.head(3)

## 2. Data cleaning

Raw problems fixed here:
- Column names differ between Kaggle versions (`batsman` vs `batter`, `innings` vs `inning`) -> standardised
- Franchises were **renamed** over the years (e.g. *Delhi Daredevils -> Delhi Capitals*) -> mapped to current names so one team = one row
- Season labels like `2007/08`, `2020/21` are inconsistent -> a clean `season_year` is derived from the match date
- Stadium names have duplicates / spelling variants -> standardised
- Over numbering normalised to 0-19
- Numeric columns coerced, missing values handled

In [ ]:
# ---- 2.1 standardise column names -------------------------------------------
matches.columns    = matches.columns.str.strip().str.lower()
deliveries.columns = deliveries.columns.str.strip().str.lower()

deliveries = deliveries.rename(columns={"batsman": "batter", "id": "match_id", "innings": "inning"})
matches    = matches.rename(columns={"match_id": "id"})

# make sure optional columns exist
for col in ["result", "result_margin", "super_over", "player_of_match", "match_type", "city"]:
    if col not in matches.columns:
        matches[col] = np.nan
for col in ["extras_type", "dismissal_kind", "player_dismissed", "fielder"]:
    if col not in deliveries.columns:
        deliveries[col] = np.nan

In [ ]:
# ---- 2.2 franchise renames -> current names ---------------------------------
TEAM_MAP = {
    "Delhi Daredevils":            "Delhi Capitals",
    "Kings XI Punjab":             "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiant":      "Rising Pune Supergiants",
    "Gujarat Lions":               "Gujarat Lions",
}
for c in ["team1", "team2", "toss_winner", "winner"]:
    matches[c] = matches[c].replace(TEAM_MAP)
for c in ["batting_team", "bowling_team"]:
    deliveries[c] = deliveries[c].replace(TEAM_MAP)

# ---- 2.3 dates and a clean season year --------------------------------------
matches["date"] = pd.to_datetime(matches["date"], errors="coerce")
matches["season_year"] = matches["date"].dt.year.astype(int)
matches["date"] = matches["date"].dt.strftime("%Y-%m-%d")

# ---- 2.4 stadium names ------------------------------------------------------
VENUE_MAP = {
    "Feroz Shah Kotla": "Arun Jaitley Stadium",
    "M.Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "Punjab Cricket Association Stadium": "Punjab Cricket Association IS Bindra Stadium",
    "Sardar Patel Stadium": "Narendra Modi Stadium",
    "Zayed Cricket Stadium": "Sheikh Zayed Stadium",
    "Rajiv Gandhi Intl Cricket Stadium": "Rajiv Gandhi International Stadium",
    "Subrata Roy Sahara Stadium": "Maharashtra Cricket Association Stadium",
    "Dr DY Patil Sports Academy": "Dr DY Patil Sports Academy",
}
matches["venue"] = (matches["venue"].astype(str).str.split(",").str[0].str.strip().replace(VENUE_MAP))

# ---- 2.5 numeric columns ----------------------------------------------------
matches["result_margin"] = pd.to_numeric(matches["result_margin"], errors="coerce")
for c in ["batsman_runs", "extra_runs", "total_runs", "is_wicket", "inning"]:
    deliveries[c] = pd.to_numeric(deliveries[c], errors="coerce").fillna(0).astype(int)

# ---- 2.6 over numbering: force 0-19 -----------------------------------------
deliveries["over"] = pd.to_numeric(deliveries["over"], errors="coerce").fillna(0).astype(int)
if deliveries["over"].min() == 1:
    deliveries["over"] -= 1

# ---- 2.7 keep only deliveries that belong to a known match ------------------
deliveries = deliveries[deliveries["match_id"].isin(matches["id"])].copy()

print("Teams in the data:", sorted(set(matches.team1) | set(matches.team2)))
print("Seasons:", matches.season_year.min(), "->", matches.season_year.max())
print("Venues after cleaning:", matches.venue.nunique())

In [ ]:
# quick data-quality summary
dq = pd.DataFrame({
    "matches_rows": [len(matches)],
    "matches_null_winner": [matches["winner"].isna().sum()],
    "deliveries_rows": [len(deliveries)],
    "duplicate_match_ids": [matches["id"].duplicated().sum()],
})
dq

## 3. Build the SQLite database

Two base tables (`matches`, `deliveries`) plus one analysis **view** (`ball_facts`) that adds
derived columns with `CASE WHEN`: legal-ball flag, balls faced, runs charged to the bowler,
bowler-credited wickets, and the innings **phase** (Powerplay / Middle / Death). Super overs are excluded.

In [ ]:
DB_PATH = "ipl.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
matches.to_sql("matches", conn, index=False)
deliveries.to_sql("deliveries", conn, index=False)

VIEW_SQL = """
DROP VIEW IF EXISTS ball_facts;
CREATE VIEW ball_facts AS
SELECT
    d.*,
    m.season_year,
    m.date AS match_date,
    -- legal delivery = not a wide / no-ball
    CASE WHEN d.extras_type IN ('wides', 'noballs') THEN 0 ELSE 1 END AS is_legal,
    -- balls faced by a batter = everything except wides
    CASE WHEN d.extras_type = 'wides' THEN 0 ELSE 1 END AS is_ball_faced,
    -- runs charged to the bowler (byes / leg-byes are not)
    d.batsman_runs
      + CASE WHEN d.extras_type IN ('wides', 'noballs') THEN d.extra_runs ELSE 0 END AS bowler_runs,
    -- wickets credited to the bowler (run-outs etc. are not)
    CASE WHEN d.is_wicket = 1
          AND d.dismissal_kind IN ('caught', 'bowled', 'lbw', 'stumped',
                                   'caught and bowled', 'hit wicket')
         THEN 1 ELSE 0 END AS bowler_wicket,
    -- innings phase (overs are 0-indexed: 0 = first over)
    CASE WHEN d."over" BETWEEN 0 AND 5  THEN 'Powerplay'
         WHEN d."over" BETWEEN 6 AND 14 THEN 'Middle'
         ELSE 'Death' END AS phase
FROM deliveries d
JOIN matches m ON m.id = d.match_id
WHERE d.inning IN (1, 2);          -- exclude super overs
"""

INDEX_SQL = """
CREATE INDEX IF NOT EXISTS idx_del_match   ON deliveries(match_id);
CREATE INDEX IF NOT EXISTS idx_del_batter  ON deliveries(batter);
CREATE INDEX IF NOT EXISTS idx_del_bowler  ON deliveries(bowler);
CREATE INDEX IF NOT EXISTS idx_m_season    ON matches(season_year);
"""

conn.executescript(INDEX_SQL)
conn.executescript(VIEW_SQL)
conn.commit()

def q(sql, key=None):
    """Run a SQL query and return a DataFrame (optionally cached in R[key])."""
    df = pd.read_sql_query(sql, conn)
    if key:
        R[key] = df
    return df

R = {}   # results store used by the charts / insights

q("SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name")

In [ ]:
q("""
SELECT 'matches' AS table_name, COUNT(*) AS row_count FROM matches
UNION ALL SELECT 'deliveries', COUNT(*) FROM deliveries
UNION ALL SELECT 'ball_facts (view)', COUNT(*) FROM ball_facts
""")

## 4. SQL analysis — 28 queries

| Section | Queries |
|---|---|
| Season and team performance | Q01 - Q05, Q10, Q11, Q27, Q28 |
| Toss and match conditions | Q06 - Q09, Q12, Q13 |
| Batting | Q14 - Q20, Q25, Q26 |
| Bowling | Q21 - Q24 |

SQL techniques used: **window functions** (`RANK`, `ROW_NUMBER`, `LAG`, running `SUM() OVER`, rolling frames), **CTEs** (including chained / nested),
**CASE WHEN**, conditional aggregation, self-joins via `UNION ALL`, gaps-and-islands, `HAVING`, views.

### Q01 — Matches, venues and results per season
**Question:** How has the tournament grown season by season?  
**Techniques:** GROUP BY, CASE WHEN

In [ ]:
Q01 = """
SELECT
    season_year,
    COUNT(*)                                                    AS matches,
    COUNT(DISTINCT venue)                                       AS venues,
    SUM(CASE WHEN result = 'no result' THEN 1 ELSE 0 END)       AS no_results,
    SUM(CASE WHEN super_over = 'Y' THEN 1 ELSE 0 END)           AS super_overs
FROM matches
GROUP BY season_year
ORDER BY season_year;
"""
q(Q01, "Q01")

### Q02 — Overall team win percentage
**Question:** Which franchises have the best win record (min. 50 matches)?  
**Techniques:** CTE, UNION ALL, CASE WHEN

In [ ]:
Q02 = """
WITH team_matches AS (
    SELECT id, team1 AS team, winner FROM matches
    UNION ALL
    SELECT id, team2 AS team, winner FROM matches
)
SELECT
    team,
    COUNT(*)                                                         AS played,
    SUM(CASE WHEN winner = team THEN 1 ELSE 0 END)                   AS wins,
    ROUND(100.0 * SUM(CASE WHEN winner = team THEN 1 ELSE 0 END)
                / COUNT(*), 2)                                       AS win_pct
FROM team_matches
GROUP BY team
HAVING COUNT(*) >= 50
ORDER BY win_pct DESC;
"""
q(Q02, "Q02")

### Q03 — Top 3 teams in every season (by wins)
**Question:** Who dominated each season?  
**Techniques:** CTE, RANK() OVER (PARTITION BY)

In [ ]:
Q03 = """
WITH season_wins AS (
    SELECT season_year, winner AS team, COUNT(*) AS wins
    FROM matches
    WHERE winner IS NOT NULL
    GROUP BY season_year, winner
),
ranked AS (
    SELECT season_year, team, wins,
           RANK() OVER (PARTITION BY season_year ORDER BY wins DESC) AS season_rank
    FROM season_wins
)
SELECT season_year, season_rank, team, wins
FROM ranked
WHERE season_rank <= 3
ORDER BY season_year, season_rank;
"""
q(Q03, "Q03")

### Q04 — Champion of every season
**Question:** Who won the title each year? (winner of the last match of the season = the final)  
**Techniques:** CTE, ROW_NUMBER() OVER (PARTITION BY)

In [ ]:
Q04 = """
WITH last_match AS (
    SELECT season_year, winner, venue, date,
           ROW_NUMBER() OVER (PARTITION BY season_year
                              ORDER BY date DESC, id DESC) AS rn
    FROM matches
)
SELECT season_year, winner AS champion, venue, date AS final_date
FROM last_match
WHERE rn = 1
ORDER BY season_year;
"""
q(Q04, "Q04")

### Q05 — Number of titles per franchise
**Question:** Which team has lifted the trophy the most?  
**Techniques:** Nested CTEs, window function, aggregation

In [ ]:
Q05 = """
WITH last_match AS (
    SELECT season_year, winner,
           ROW_NUMBER() OVER (PARTITION BY season_year
                              ORDER BY date DESC, id DESC) AS rn
    FROM matches
),
champions AS (
    SELECT season_year, winner FROM last_match WHERE rn = 1 AND winner IS NOT NULL
)
SELECT winner AS team,
       COUNT(*)                          AS titles,
       GROUP_CONCAT(season_year, ', ')   AS seasons
FROM champions
GROUP BY winner
ORDER BY titles DESC, team;
"""
q(Q05, "Q05")

### Q06 — Does winning the toss help?
**Question:** How often does the toss winner also win the match, by toss decision?  
**Techniques:** CASE WHEN, conditional aggregation

In [ ]:
Q06 = """
SELECT
    toss_decision,
    COUNT(*)                                                       AS matches,
    SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END)          AS toss_winner_won,
    ROUND(100.0 * SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END)
                / COUNT(*), 2)                                     AS toss_win_match_win_pct
FROM matches
WHERE winner IS NOT NULL
GROUP BY toss_decision;
"""
q(Q06, "Q06")

### Q07 — Toss decision trend
**Question:** Have captains shifted from batting first to fielding first?  
**Techniques:** CASE WHEN, percentage by season

In [ ]:
Q07 = """
SELECT
    season_year,
    COUNT(*)                                                           AS matches,
    ROUND(100.0 * SUM(CASE WHEN toss_decision = 'field' THEN 1 ELSE 0 END)
                / COUNT(*), 1)                                         AS pct_chose_field,
    ROUND(100.0 * SUM(CASE WHEN toss_decision = 'bat' THEN 1 ELSE 0 END)
                / COUNT(*), 1)                                         AS pct_chose_bat
FROM matches
GROUP BY season_year
ORDER BY season_year;
"""
q(Q07, "Q07")

### Q08 — Chasing vs defending
**Question:** Is it easier to chase or to defend? (win share by season)  
**Techniques:** CASE WHEN, conditional aggregation

In [ ]:
Q08 = """
SELECT
    season_year,
    SUM(CASE WHEN result = 'wickets' THEN 1 ELSE 0 END)  AS chasing_wins,
    SUM(CASE WHEN result = 'runs'    THEN 1 ELSE 0 END)  AS defending_wins,
    ROUND(100.0 * SUM(CASE WHEN result = 'wickets' THEN 1 ELSE 0 END)
                / COUNT(*), 1)                           AS chase_win_pct
FROM matches
WHERE result IN ('runs', 'wickets')
GROUP BY season_year
ORDER BY season_year;
"""
q(Q08, "Q08")

### Q09 — Venue scoring profile
**Question:** Which grounds are the highest scoring? (min. 20 matches)  
**Techniques:** CTE, JOIN, conditional AVG

In [ ]:
Q09 = """
WITH inn_totals AS (
    SELECT match_id, inning, SUM(total_runs) AS runs
    FROM deliveries
    WHERE inning IN (1, 2)
    GROUP BY match_id, inning
)
SELECT
    m.venue,
    COUNT(DISTINCT m.id)                                   AS matches,
    ROUND(AVG(CASE WHEN t.inning = 1 THEN t.runs END), 1)  AS avg_1st_innings,
    ROUND(AVG(CASE WHEN t.inning = 2 THEN t.runs END), 1)  AS avg_2nd_innings
FROM matches m
JOIN inn_totals t ON t.match_id = m.id
GROUP BY m.venue
HAVING COUNT(DISTINCT m.id) >= 20
ORDER BY avg_1st_innings DESC
LIMIT 15;
"""
q(Q09, "Q09")

### Q10 — Head-to-head records of the biggest rivalries
**Question:** Who leads the most-played match-ups?  
**Techniques:** CTE, CASE WHEN to normalise team pairs

In [ ]:
Q10 = """
WITH pairs AS (
    SELECT
        CASE WHEN team1 < team2 THEN team1 ELSE team2 END AS team_a,
        CASE WHEN team1 < team2 THEN team2 ELSE team1 END AS team_b,
        winner
    FROM matches
)
SELECT
    team_a,
    team_b,
    COUNT(*)                                          AS matches,
    SUM(CASE WHEN winner = team_a THEN 1 ELSE 0 END)  AS team_a_wins,
    SUM(CASE WHEN winner = team_b THEN 1 ELSE 0 END)  AS team_b_wins
FROM pairs
GROUP BY team_a, team_b
ORDER BY matches DESC
LIMIT 15;
"""
q(Q10, "Q10")

### Q11 — Longest winning streaks (gaps-and-islands)
**Question:** What are the longest consecutive-win runs by any team?  
**Techniques:** UNION ALL, ROW_NUMBER, gaps-and-islands pattern

In [ ]:
Q11 = """
WITH team_matches AS (
    SELECT id, date, team1 AS team, winner FROM matches
    UNION ALL
    SELECT id, date, team2 AS team, winner FROM matches
),
ordered AS (
    SELECT team, date, id,
           CASE WHEN winner = team THEN 1 ELSE 0 END AS won,
           ROW_NUMBER() OVER (PARTITION BY team ORDER BY date, id) AS rn
    FROM team_matches
),
wins_only AS (
    SELECT team, date,
           rn - ROW_NUMBER() OVER (PARTITION BY team ORDER BY rn) AS grp
    FROM ordered
    WHERE won = 1
)
SELECT team,
       COUNT(*)  AS streak_length,
       MIN(date) AS streak_start,
       MAX(date) AS streak_end
FROM wins_only
GROUP BY team, grp
ORDER BY streak_length DESC
LIMIT 10;
"""
q(Q11, "Q11")

### Q12 — Biggest wins by runs
**Question:** Which were the most one-sided matches when defending?  
**Techniques:** CASE WHEN, ORDER BY / LIMIT

In [ ]:
Q12 = """
SELECT
    season_year,
    date,
    winner,
    CASE WHEN winner = team1 THEN team2 ELSE team1 END AS loser,
    result_margin                                     AS won_by_runs,
    venue
FROM matches
WHERE result = 'runs'
ORDER BY result_margin DESC
LIMIT 10;
"""
q(Q12, "Q12")

### Q13 — Nail-biters: who wins the close ones?
**Question:** Which teams win the most matches decided by <=10 runs or <=2 wickets?  
**Techniques:** CASE-style filtering, aggregation

In [ ]:
Q13 = """
SELECT
    winner AS team,
    COUNT(*) AS close_wins,
    SUM(CASE WHEN result = 'runs'    THEN 1 ELSE 0 END) AS close_wins_defending,
    SUM(CASE WHEN result = 'wickets' THEN 1 ELSE 0 END) AS close_wins_chasing
FROM matches
WHERE (result = 'runs'    AND result_margin <= 10)
   OR (result = 'wickets' AND result_margin <= 2)
GROUP BY winner
ORDER BY close_wins DESC
LIMIT 10;
"""
q(Q13, "Q13")

### Q14 — Player of the Match leaders
**Question:** Who has won the most Player of the Match awards?  
**Techniques:** RANK() OVER, aggregation

In [ ]:
Q14 = """
WITH awards AS (
    SELECT player_of_match AS player, COUNT(*) AS awards
    FROM matches
    WHERE player_of_match IS NOT NULL
    GROUP BY player_of_match
)
SELECT player, awards,
       RANK() OVER (ORDER BY awards DESC) AS award_rank
FROM awards
ORDER BY awards DESC, player
LIMIT 10;
"""
q(Q14, "Q14")

### Q15 — All-time top run scorers (with strike rate and average)
**Question:** Who are the leading run scorers and how efficient are they?  
**Techniques:** CTEs, JOIN, view, conditional aggregation

In [ ]:
Q15 = """
WITH bat AS (
    SELECT
        batter,
        COUNT(DISTINCT match_id)                           AS innings,
        SUM(batsman_runs)                                  AS runs,
        SUM(is_ball_faced)                                 AS balls,
        SUM(CASE WHEN batsman_runs = 4 THEN 1 ELSE 0 END)  AS fours,
        SUM(CASE WHEN batsman_runs = 6 THEN 1 ELSE 0 END)  AS sixes
    FROM ball_facts
    GROUP BY batter
),
outs AS (
    SELECT player_dismissed AS batter, COUNT(*) AS dismissals
    FROM ball_facts
    WHERE is_wicket = 1 AND dismissal_kind <> 'retired hurt'
    GROUP BY player_dismissed
)
SELECT
    b.batter, b.innings, b.runs, b.balls,
    ROUND(100.0 * b.runs / b.balls, 2)                 AS strike_rate,
    ROUND(1.0 * b.runs / NULLIF(o.dismissals, 0), 2)   AS average,
    b.fours, b.sixes
FROM bat b
LEFT JOIN outs o ON o.batter = b.batter
ORDER BY b.runs DESC
LIMIT 10;
"""
q(Q15, "Q15")

### Q16 — Orange Cap: top run scorer of every season
**Question:** Who finished as the leading run scorer each year?  
**Techniques:** CTE, RANK() OVER (PARTITION BY)

In [ ]:
Q16 = """
WITH season_runs AS (
    SELECT season_year, batter, SUM(batsman_runs) AS runs
    FROM ball_facts
    GROUP BY season_year, batter
),
ranked AS (
    SELECT *, RANK() OVER (PARTITION BY season_year ORDER BY runs DESC) AS rk
    FROM season_runs
)
SELECT season_year, batter AS orange_cap, runs
FROM ranked
WHERE rk = 1
ORDER BY season_year;
"""
q(Q16, "Q16")

### Q17 — Career run progression of the top 5 batters
**Question:** How did the leading batters accumulate their runs season by season?  
**Techniques:** CTE, subquery, SUM() OVER (running total)

In [ ]:
Q17 = """
WITH top5 AS (
    SELECT batter
    FROM ball_facts
    GROUP BY batter
    ORDER BY SUM(batsman_runs) DESC
    LIMIT 5
),
season_runs AS (
    SELECT batter, season_year, SUM(batsman_runs) AS runs
    FROM ball_facts
    WHERE batter IN (SELECT batter FROM top5)
    GROUP BY batter, season_year
)
SELECT
    batter,
    season_year,
    runs,
    SUM(runs) OVER (PARTITION BY batter ORDER BY season_year) AS cumulative_runs
FROM season_runs
ORDER BY batter, season_year;
"""
q(Q17, "Q17")

### Q18 — Centuries and fifties
**Question:** Who has the most 100s and 50s in IPL history?  
**Techniques:** CTE, CASE WHEN banding

In [ ]:
Q18 = """
WITH innings_scores AS (
    SELECT match_id, batter, SUM(batsman_runs) AS runs
    FROM ball_facts
    GROUP BY match_id, batter
)
SELECT
    batter,
    SUM(CASE WHEN runs >= 100            THEN 1 ELSE 0 END) AS hundreds,
    SUM(CASE WHEN runs BETWEEN 50 AND 99 THEN 1 ELSE 0 END) AS fifties,
    MAX(runs)                                               AS highest_score
FROM innings_scores
GROUP BY batter
HAVING MAX(runs) >= 50
ORDER BY hundreds DESC, fifties DESC
LIMIT 10;
"""
q(Q18, "Q18")

### Q19 — Run rate and wickets by innings phase and season
**Question:** How has scoring changed in the powerplay, middle overs and death overs?  
**Techniques:** CASE WHEN (derived phase), conditional aggregation

In [ ]:
Q19 = """
SELECT
    season_year,
    phase,
    SUM(total_runs)                                   AS runs,
    SUM(is_legal)                                     AS legal_balls,
    ROUND(6.0 * SUM(total_runs) / SUM(is_legal), 2)   AS run_rate,
    SUM(is_wicket)                                    AS wickets
FROM ball_facts
GROUP BY season_year, phase
ORDER BY season_year,
         CASE phase WHEN 'Powerplay' THEN 1 WHEN 'Middle' THEN 2 ELSE 3 END;
"""
q(Q19, "Q19")

### Q20 — Death-over hitters
**Question:** Which batters score fastest in overs 16-20? (min. 300 balls)  
**Techniques:** Filtered aggregation, HAVING

In [ ]:
Q20 = """
SELECT
    batter,
    SUM(batsman_runs)                                       AS runs,
    SUM(is_ball_faced)                                      AS balls,
    ROUND(100.0 * SUM(batsman_runs) / SUM(is_ball_faced), 2) AS death_strike_rate
FROM ball_facts
WHERE phase = 'Death'
GROUP BY batter
HAVING SUM(is_ball_faced) >= 300
ORDER BY death_strike_rate DESC
LIMIT 10;
"""
q(Q20, "Q20")

### Q21 — Top wicket takers (economy, average, strike rate)
**Question:** Who are the leading bowlers and how economical are they?  
**Techniques:** Aggregation over derived columns

In [ ]:
Q21 = """
SELECT
    bowler,
    SUM(bowler_wicket)                                                AS wickets,
    SUM(is_legal)                                                     AS legal_balls,
    ROUND(6.0 * SUM(bowler_runs) / SUM(is_legal), 2)                  AS economy,
    ROUND(1.0 * SUM(bowler_runs) / NULLIF(SUM(bowler_wicket), 0), 2)  AS bowling_average,
    ROUND(1.0 * SUM(is_legal)   / NULLIF(SUM(bowler_wicket), 0), 1)   AS bowling_strike_rate
FROM ball_facts
GROUP BY bowler
ORDER BY wickets DESC
LIMIT 10;
"""
q(Q21, "Q21")

### Q22 — Purple Cap: top wicket taker of every season
**Question:** Who led the wicket charts each year?  
**Techniques:** CTE, RANK() OVER (PARTITION BY)

In [ ]:
Q22 = """
WITH season_wkts AS (
    SELECT season_year, bowler, SUM(bowler_wicket) AS wickets
    FROM ball_facts
    GROUP BY season_year, bowler
),
ranked AS (
    SELECT *, RANK() OVER (PARTITION BY season_year ORDER BY wickets DESC) AS rk
    FROM season_wkts
)
SELECT season_year, bowler AS purple_cap, wickets
FROM ranked
WHERE rk = 1
ORDER BY season_year;
"""
q(Q22, "Q22")

### Q23 — Death-over bowling specialists
**Question:** Who bowls the most economical death overs? (min. 300 balls)  
**Techniques:** Filtered aggregation, HAVING

In [ ]:
Q23 = """
SELECT
    bowler,
    SUM(is_legal)                                         AS legal_balls,
    SUM(bowler_wicket)                                    AS wickets,
    ROUND(6.0 * SUM(bowler_runs) / SUM(is_legal), 2)      AS death_economy
FROM ball_facts
WHERE phase = 'Death'
GROUP BY bowler
HAVING SUM(is_legal) >= 300
ORDER BY death_economy ASC
LIMIT 10;
"""
q(Q23, "Q23")

### Q24 — How do batters get out?
**Question:** What share of dismissals does each mode account for?  
**Techniques:** Window function over the whole result set: SUM(COUNT(*)) OVER ()

In [ ]:
Q24 = """
SELECT
    dismissal_kind,
    COUNT(*)                                            AS dismissals,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)  AS pct_of_total
FROM ball_facts
WHERE is_wicket = 1 AND dismissal_kind IS NOT NULL
GROUP BY dismissal_kind
ORDER BY dismissals DESC;
"""
q(Q24, "Q24")

### Q25 — Highest team totals
**Question:** What are the biggest innings totals ever posted?  
**Techniques:** CTE, JOIN

In [ ]:
Q25 = """
WITH totals AS (
    SELECT match_id, inning, batting_team, bowling_team,
           SUM(total_runs) AS runs,
           SUM(is_wicket)  AS wickets_lost
    FROM deliveries
    WHERE inning IN (1, 2)
    GROUP BY match_id, inning, batting_team, bowling_team
)
SELECT t.batting_team, t.bowling_team AS against, t.runs, t.wickets_lost,
       m.season_year, m.venue
FROM totals t
JOIN matches m ON m.id = t.match_id
ORDER BY t.runs DESC
LIMIT 10;
"""
q(Q25, "Q25")

### Q26 — Powerplay score vs. winning chances
**Question:** Does a big powerplay translate into wins?  
**Techniques:** CTEs, CASE WHEN banding, AVG of a flag

In [ ]:
Q26 = """
WITH pp AS (
    SELECT match_id, batting_team AS team, SUM(total_runs) AS pp_runs
    FROM deliveries
    WHERE inning IN (1, 2) AND "over" <= 5
    GROUP BY match_id, batting_team
),
flagged AS (
    SELECT pp.pp_runs,
           CASE WHEN m.winner = pp.team THEN 1 ELSE 0 END AS won,
           CASE WHEN pp.pp_runs >= 60 THEN '4) 60+'
                WHEN pp.pp_runs >= 50 THEN '3) 50-59'
                WHEN pp.pp_runs >= 40 THEN '2) 40-49'
                ELSE '1) under 40' END                    AS pp_band
    FROM pp
    JOIN matches m ON m.id = pp.match_id
    WHERE m.winner IS NOT NULL
)
SELECT pp_band,
       COUNT(*)                     AS innings,
       ROUND(100.0 * AVG(won), 1)   AS win_pct
FROM flagged
GROUP BY pp_band
ORDER BY pp_band;
"""
q(Q26, "Q26")

### Q27 — Bounce-back ability
**Question:** Which teams recover best after a defeat? (win % after a loss vs after a win)  
**Techniques:** CTEs, LAG() OVER (PARTITION BY ... ORDER BY ...)

In [ ]:
Q27 = """
WITH team_matches AS (
    SELECT id, date, team1 AS team, winner FROM matches
    UNION ALL
    SELECT id, date, team2 AS team, winner FROM matches
),
seq AS (
    SELECT team,
           CASE WHEN winner = team THEN 1 ELSE 0 END AS won,
           LAG(CASE WHEN winner = team THEN 1 ELSE 0 END)
               OVER (PARTITION BY team ORDER BY date, id) AS prev_won
    FROM team_matches
)
SELECT
    team,
    COUNT(*) AS matches,
    ROUND(100.0 * AVG(CASE WHEN prev_won = 0 THEN won END), 1) AS win_pct_after_loss,
    ROUND(100.0 * AVG(CASE WHEN prev_won = 1 THEN won END), 1) AS win_pct_after_win
FROM seq
WHERE prev_won IS NOT NULL
GROUP BY team
HAVING COUNT(*) >= 50
ORDER BY win_pct_after_loss DESC;
"""
q(Q27, "Q27")

### Q28 — Rolling 5-match form (peak form per team)
**Question:** What was each team's best 5-match run of form?  
**Techniques:** SUM() OVER with ROWS BETWEEN frame, CTE

In [ ]:
Q28 = """
WITH team_matches AS (
    SELECT id, date, team1 AS team, winner FROM matches
    UNION ALL
    SELECT id, date, team2 AS team, winner FROM matches
),
form AS (
    SELECT team, date,
           SUM(CASE WHEN winner = team THEN 1 ELSE 0 END)
               OVER (PARTITION BY team ORDER BY date, id
                     ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS wins_last_5,
           ROW_NUMBER() OVER (PARTITION BY team ORDER BY date, id) AS match_no
    FROM team_matches
)
SELECT team,
       MAX(wins_last_5) AS best_5_match_wins,
       COUNT(CASE WHEN wins_last_5 = 5 THEN 1 END) AS five_from_five_windows
FROM form
WHERE match_no >= 5
GROUP BY team
HAVING COUNT(*) >= 30
ORDER BY five_from_five_windows DESC, team;
"""
q(Q28, "Q28")

## 5. Visualisations

In [ ]:
def savefig(name):
    plt.tight_layout()
    plt.savefig(f"images/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

# 1 - Matches per season
d = R["Q01"]
plt.bar(d.season_year.astype(str), d.matches, color="#1f77b4")
plt.title("Matches played per IPL season"); plt.xlabel("Season"); plt.ylabel("Matches")
plt.xticks(rotation=45)
savefig("01_matches_per_season")

In [ ]:
# 2 - Team win %
d = R["Q02"].sort_values("win_pct")
plt.barh(d.team, d.win_pct, color="#2ca02c")
plt.axvline(50, color="grey", ls="--", lw=1)
plt.title("Overall win % by franchise (min. 50 matches)"); plt.xlabel("Win %")
for i, v in enumerate(d.win_pct):
    plt.text(v + 0.3, i, f"{v:.1f}", va="center", fontsize=8)
savefig("02_team_win_pct")

In [ ]:
# 3 - Titles
d = R["Q05"].sort_values("titles")
plt.barh(d.team, d.titles, color="#ff7f0e")
plt.title("IPL titles by franchise"); plt.xlabel("Titles")
savefig("03_titles")

In [ ]:
# 4 - Toss decision trend
d = R["Q07"]
plt.plot(d.season_year, d.pct_chose_field, marker="o", label="Chose to field", color="#d62728")
plt.plot(d.season_year, d.pct_chose_bat,   marker="o", label="Chose to bat",   color="#1f77b4")
plt.title("Toss decision trend: bat first vs field first"); plt.xlabel("Season"); plt.ylabel("% of matches")
plt.legend()
savefig("04_toss_trend")

In [ ]:
# 5 - Top run scorers
d = R["Q15"].sort_values("runs")
plt.barh(d.batter, d.runs, color="#9467bd")
for i, (r, sr) in enumerate(zip(d.runs, d.strike_rate)):
    plt.text(r + 20, i, f"SR {sr}", va="center", fontsize=8)
plt.title("Top 10 run scorers in IPL history"); plt.xlabel("Runs")
savefig("05_top_run_scorers")

In [ ]:
# 6 - Top wicket takers
d = R["Q21"].sort_values("wickets")
plt.barh(d.bowler, d.wickets, color="#8c564b")
for i, (w, e) in enumerate(zip(d.wickets, d.economy)):
    plt.text(w + 1, i, f"Econ {e}", va="center", fontsize=8)
plt.title("Top 10 wicket takers in IPL history"); plt.xlabel("Wickets")
savefig("06_top_wicket_takers")

In [ ]:
# 7 - Run rate by phase over the seasons
d = R["Q19"].pivot(index="season_year", columns="phase", values="run_rate")[["Powerplay", "Middle", "Death"]]
d.plot(marker="o", ax=plt.gca())
plt.title("Run rate by innings phase"); plt.xlabel("Season"); plt.ylabel("Runs per over")
savefig("07_phase_run_rate")

In [ ]:
# 8 - Powerplay vs winning
d = R["Q26"]
plt.bar(d.pp_band, d.win_pct, color="#17becf")
plt.axhline(50, color="grey", ls="--", lw=1)
plt.title("Win % by powerplay score"); plt.ylabel("Win %"); plt.xlabel("Powerplay runs")
for i, v in enumerate(d.win_pct):
    plt.text(i, v + 0.8, f"{v}%", ha="center")
savefig("08_powerplay_vs_win")

In [ ]:
# 9 - Running career runs for the top 5 batters
d = R["Q17"].pivot(index="season_year", columns="batter", values="cumulative_runs").ffill()
d.plot(marker="o", ax=plt.gca())
plt.title("Career run progression - top 5 run scorers"); plt.xlabel("Season"); plt.ylabel("Cumulative runs")
savefig("09_career_progression")

## 6. Key insights (auto-generated from the query results)

In [ ]:
def top(df, col):
    return df.iloc[0][col]

ins = []
ins.append(f"Best overall win rate (min. 50 matches): {top(R['Q02'], 'team')} at {top(R['Q02'], 'win_pct')}%.")
ins.append(f"Most titles: {top(R['Q05'], 'team')} ({top(R['Q05'], 'titles')}).")

t = R["Q06"]
ins.append("Toss winners win the match " + ", ".join(
    f"{r.toss_win_match_win_pct}% of the time when they {r.toss_decision}" for r in t.itertuples()) + ".")

c = R["Q08"]
ins.append(f"Chasing sides have won on average {c.chase_win_pct.mean():.1f}% of decided matches across seasons.")

f = R["Q07"]
ins.append(f"Captains chose to field first in {f.pct_chose_field.iloc[0]}% of toss decisions in {f.season_year.iloc[0]} vs {f.pct_chose_field.iloc[-1]}% in {f.season_year.iloc[-1]}.")

v = R["Q09"]
ins.append(f"Highest-scoring venue (first innings avg): {top(v, 'venue')} - {top(v, 'avg_1st_innings')} runs.")

ins.append(f"Leading run scorer: {top(R['Q15'], 'batter')} with {top(R['Q15'], 'runs')} runs. "
           f"Leading wicket taker: {top(R['Q21'], 'bowler')} with {top(R['Q21'], 'wickets')} wickets.")

p = R["Q26"]
ins.append(f"Teams scoring 60+ in the powerplay win {p.win_pct.iloc[-1]}% of matches vs {p.win_pct.iloc[0]}% when scoring under 40.")

s = R["Q11"]
ins.append(f"Longest winning streak: {top(s, 'team')} with {top(s, 'streak_length')} consecutive wins.")

print("\n".join(f"{i+1}. {x}" for i, x in enumerate(ins)))

## 7. Export tidy CSVs for Tableau Public

Each file has one clear grain so it can be dragged straight into Tableau.

In [ ]:
EXPORTS = {
    "01_season_summary": """
SELECT season_year, COUNT(*) AS matches, COUNT(DISTINCT venue) AS venues,
       SUM(CASE WHEN super_over = 'Y' THEN 1 ELSE 0 END) AS super_overs
FROM matches GROUP BY season_year ORDER BY season_year;
""",
    "02_team_season_performance": """
WITH tm AS (
    SELECT season_year, team1 AS team, winner FROM matches
    UNION ALL
    SELECT season_year, team2 AS team, winner FROM matches
)
SELECT season_year, team,
       COUNT(*) AS played,
       SUM(CASE WHEN winner = team THEN 1 ELSE 0 END) AS wins,
       ROUND(100.0 * SUM(CASE WHEN winner = team THEN 1 ELSE 0 END) / COUNT(*), 1) AS win_pct
FROM tm GROUP BY season_year, team ORDER BY season_year, wins DESC;
""",
    "03_toss_by_season": """
SELECT season_year, toss_decision, COUNT(*) AS matches,
       SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END) AS toss_winner_won
FROM matches WHERE winner IS NOT NULL
GROUP BY season_year, toss_decision ORDER BY season_year;
""",
    "04_top_batters": """
WITH bat AS (
    SELECT batter, COUNT(DISTINCT match_id) AS innings, SUM(batsman_runs) AS runs,
           SUM(is_ball_faced) AS balls,
           SUM(CASE WHEN batsman_runs = 4 THEN 1 ELSE 0 END) AS fours,
           SUM(CASE WHEN batsman_runs = 6 THEN 1 ELSE 0 END) AS sixes
    FROM ball_facts GROUP BY batter
),
outs AS (
    SELECT player_dismissed AS batter, COUNT(*) AS dismissals
    FROM ball_facts WHERE is_wicket = 1 AND dismissal_kind <> 'retired hurt'
    GROUP BY player_dismissed
)
SELECT b.batter, b.innings, b.runs, b.balls,
       ROUND(100.0 * b.runs / b.balls, 2) AS strike_rate,
       ROUND(1.0 * b.runs / NULLIF(o.dismissals, 0), 2) AS average,
       b.fours, b.sixes
FROM bat b LEFT JOIN outs o ON o.batter = b.batter
WHERE b.balls >= 300
ORDER BY b.runs DESC LIMIT 100;
""",
    "05_top_bowlers": """
SELECT bowler, SUM(bowler_wicket) AS wickets, SUM(is_legal) AS legal_balls,
       ROUND(6.0 * SUM(bowler_runs) / SUM(is_legal), 2) AS economy,
       ROUND(1.0 * SUM(is_legal) / NULLIF(SUM(bowler_wicket), 0), 1) AS bowling_strike_rate
FROM ball_facts GROUP BY bowler HAVING SUM(is_legal) >= 300
ORDER BY wickets DESC LIMIT 100;
""",
    "06_venue_stats": """
WITH inn_totals AS (
    SELECT match_id, inning, SUM(total_runs) AS runs
    FROM deliveries WHERE inning IN (1, 2) GROUP BY match_id, inning
)
SELECT m.venue, MAX(m.city) AS city, COUNT(DISTINCT m.id) AS matches,
       ROUND(AVG(CASE WHEN t.inning = 1 THEN t.runs END), 1) AS avg_1st_innings,
       ROUND(AVG(CASE WHEN t.inning = 2 THEN t.runs END), 1) AS avg_2nd_innings
FROM matches m JOIN inn_totals t ON t.match_id = m.id
GROUP BY m.venue HAVING COUNT(DISTINCT m.id) >= 5 ORDER BY matches DESC;
""",
    "07_phase_run_rate": """
SELECT season_year, phase, SUM(total_runs) AS runs, SUM(is_legal) AS legal_balls,
       ROUND(6.0 * SUM(total_runs) / SUM(is_legal), 2) AS run_rate, SUM(is_wicket) AS wickets
FROM ball_facts GROUP BY season_year, phase ORDER BY season_year;
""",
    "08_season_caps": """
WITH bat AS (
    SELECT season_year, batter AS player, SUM(batsman_runs) AS value, 'Orange Cap (runs)' AS award,
           RANK() OVER (PARTITION BY season_year ORDER BY SUM(batsman_runs) DESC) AS rk
    FROM ball_facts GROUP BY season_year, batter
),
bowl AS (
    SELECT season_year, bowler AS player, SUM(bowler_wicket) AS value, 'Purple Cap (wickets)' AS award,
           RANK() OVER (PARTITION BY season_year ORDER BY SUM(bowler_wicket) DESC) AS rk
    FROM ball_facts GROUP BY season_year, bowler
)
SELECT season_year, award, player, value FROM bat WHERE rk = 1
UNION ALL
SELECT season_year, award, player, value FROM bowl WHERE rk = 1
ORDER BY season_year, award;
""",
    "09_matches_clean": """
SELECT id AS match_id, season_year, date, city, venue, team1, team2, toss_winner,
       toss_decision, winner, result, result_margin, player_of_match, super_over
FROM matches ORDER BY date, id;
"""
}

for name, sql in EXPORTS.items():
    df = pd.read_sql_query(sql, conn)
    df.to_csv(f"tableau_exports/{name}.csv", index=False)
    print(f"{name}.csv  ->  {len(df):>6} rows x {df.shape[1]} cols")

## 8. Package and download

In [ ]:
shutil.make_archive("ipl_tableau_exports", "zip", "tableau_exports")
shutil.make_archive("ipl_charts", "zip", "images")

try:
    from google.colab import files
    files.download("ipl_tableau_exports.zip")   # CSVs for Tableau
    files.download("ipl_charts.zip")            # PNG charts for the README
    files.download("ipl.db")                    # the SQLite database
except ImportError:
    print("Not running in Colab - files are saved in the working directory.")

## 9. Conclusion

* SQL (SQLite) was used end-to-end to turn 1,000+ matches and 250k+ ball-by-ball records into team, batter, bowler and venue insights using **28 queries** with window functions, CTEs and CASE WHEN logic.
* Cleaned and standardised franchise names, venues, seasons and over numbering so metrics are consistent across every season.
* Tidy exports power an interactive **Tableau Public** dashboard (see `README.md` for the build guide).

**Next steps:** add player-vs-player matchups, win-probability modelling with Scikit-learn, and a Streamlit front end.